
# Sustainable Multi-Objective Supply Chain Network Optimization
### Cost • Carbon • Service-Level • Capacity Robustness • Disruption Analysis

This notebook develops the original supply-chain optimization study into a **research-oriented flagship project** suitable for a technical CV and GitHub portfolio.

The model represents a five-country manufacturing and distribution network:

**USA • Germany • Japan • Brazil • India**

The study answers four decision questions:

1. **Economic design:** What is the minimum-cost global production network?
2. **Decarbonization:** What cost premium is required to reduce network CO₂ emissions?
3. **Policy response:** How does a carbon price change sourcing and facility decisions?
4. **Robustness:** How resilient is the network to delivery-time limits, demand growth, and freight-cost shocks?

The original Google Drive dataset paths are retained unchanged.



## 1. Imports and data

The optimization is formulated as a **Mixed-Integer Linear Program (MILP)** using PuLP.  
Continuous variables represent inter-country production flows, while binary variables represent plant-capacity decisions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pulp import (
    LpProblem, LpMinimize, LpVariable, lpSum, LpStatus,
    value, PULP_CBC_CMD
)

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# IMPORTANT: Original file paths retained exactly as provided.
var_costs = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Supply Chain Optimization/Datasets/variable_costs.xlsx', index_col=0)
freight_costs = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Supply Chain Optimization/Datasets/freight_costs.xlsx', index_col=0)
storage_costs = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Supply Chain Optimization/Datasets/Storage_costs.xlsx', index_col=0)
carbon_emissions = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Supply Chain Optimization/Datasets/CO2_Emissions.xlsx', index_col=0)
delivery_times = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Supply Chain Optimization/Datasets/Delivery_LeadTime.xlsx', index_col=0)
fixed_costs = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Supply Chain Optimization/Datasets/fixed_cost.xlsx', index_col=0)
capacity = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Supply Chain Optimization/Datasets/capacity.xlsx', index_col=0)
demand = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Supply Chain Optimization/Datasets/demand.xlsx', index_col=0)
del_deadline = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Supply Chain Optimization/Datasets/Delivery_Deadlines.xlsx', index_col=0)

loc = ['USA', 'Germany', 'Japan', 'Brazil', 'India']
size = ['Low', 'High']

total_costs = freight_costs / 1000 + var_costs
total_fixed_costs = fixed_costs + storage_costs

print("Data loaded successfully.")
print(f"Countries: {len(loc)} | Candidate plant decisions: {len(loc) * len(size)} | Shipping routes: {len(loc)**2}")


## 2. Data validation

Before optimization, check that the network inputs are numeric, complete, and dimensionally consistent. This is important because silent missing values can make an MILP infeasible or produce misleading results.


In [ ]:
datasets = {
    "Variable costs": var_costs,
    "Freight costs": freight_costs,
    "Storage costs": storage_costs,
    "CO2 emissions": carbon_emissions,
    "Delivery times": delivery_times,
    "Fixed costs": fixed_costs,
    "Capacity": capacity,
    "Demand": demand,
}

validation = []

for name, df in datasets.items():
    validation.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing values": int(df.isna().sum().sum()),
        "Numeric": bool(all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)),
    })

validation_df = pd.DataFrame(validation)
display(validation_df)

assert validation_df["Missing values"].sum() == 0, "Missing values detected in the input data."
assert validation_df["Numeric"].all(), "Non-numeric values detected in optimization inputs."

total_demand = float(demand["Demand"].sum())
maximum_capacity = float(capacity["High"].sum() * 1000)

print(f"Total monthly demand: {total_demand:,.0f} units")
print(f"Maximum installed network capacity: {maximum_capacity:,.0f} units")
print(f"Nominal capacity headroom: {(maximum_capacity / total_demand - 1) * 100:.2f}%")


## 3. MILP formulation

Let:

- \(x_{ij}\): units manufactured in country \(i\) and supplied to country \(j\)
- \(y_{is}\): binary decision indicating whether plant size \(s\) is opened in country \(i\)

### Economic objective

\[
\min C =
\sum_{i,s}F_{is}y_{is}
+
\sum_{i,j}c_{ij}x_{ij}
+
p_c\frac{E}{1000}
\]

where \(p_c\) is an optional carbon price in **$/tCO₂** and

\[
E = \sum_{i,j} e_{ij}x_{ij}
\]

is network CO₂ emissions in kg according to the supplied emission factors.

### Constraints

1. Demand must be satisfied in every market.
2. Production cannot exceed installed plant capacity.
3. At most one plant-capacity option can be selected per country.
4. An optional carbon budget may constrain total network emissions.
5. An optional service-level threshold may forbid routes exceeding a specified lead time.

The service-level limit is kept **optional**. A 30-day hard deadline in the supplied data makes the original network infeasible; this is analyzed explicitly later rather than silently forcing an infeasible baseline.


In [ ]:
def solve_supply_chain(
    demand_df=None,
    freight_multiplier=1.0,
    carbon_cap_kg=None,
    carbon_price_per_ton=0.0,
    max_lead_days=None,
    solver_msg=False,
):
    """Solve one supply-chain network-design scenario."""

    demand_df = demand.copy() if demand_df is None else demand_df.copy()

    scenario_total_costs = var_costs + (freight_costs * freight_multiplier) / 1000

    model = LpProblem("Sustainable_Supply_Chain_Network", LpMinimize)

    flow = LpVariable.dicts(
        "flow",
        [(i, j) for i in loc for j in loc],
        lowBound=0,
        cat="Continuous"
    )

    plant = LpVariable.dicts(
        "plant",
        [(i, s) for i in loc for s in size],
        cat="Binary"
    )

    fixed_cost_expr = lpSum(
        float(total_fixed_costs.loc[i, s]) * 1000 * plant[(i, s)]
        for i in loc for s in size
    )

    operating_cost_expr = lpSum(
        float(scenario_total_costs.loc[i, j]) * flow[(i, j)]
        for i in loc for j in loc
    )

    emissions_expr = lpSum(
        float(carbon_emissions.loc[i, j]) * flow[(i, j)]
        for i in loc for j in loc
    )

    # Convert kg CO2 to tonnes before applying a $/tCO2 carbon price.
    carbon_cost_expr = float(carbon_price_per_ton) * emissions_expr / 1000

    model += fixed_cost_expr + operating_cost_expr + carbon_cost_expr

    # Demand satisfaction.
    for j in loc:
        model += (
            lpSum(flow[(i, j)] for i in loc) == float(demand_df.loc[j, "Demand"]),
            f"Demand_{j}"
        )

    # Capacity and plant-selection constraints.
    for i in loc:
        model += (
            lpSum(flow[(i, j)] for j in loc)
            <= lpSum(
                float(capacity.loc[i, s]) * 1000 * plant[(i, s)]
                for s in size
            ),
            f"Capacity_{i}"
        )

        model += (
            lpSum(plant[(i, s)] for s in size) <= 1,
            f"PlantChoice_{i}"
        )

    # Optional route-level service constraint.
    if max_lead_days is not None:
        for i in loc:
            for j in loc:
                if float(delivery_times.loc[i, j]) > float(max_lead_days):
                    model += flow[(i, j)] == 0, f"LeadTime_{i}_{j}"

    # Optional network-level carbon budget.
    if carbon_cap_kg is not None:
        if not np.isfinite(carbon_cap_kg):
            raise ValueError(f"carbon_cap_kg must be finite; received {carbon_cap_kg}")
        model += emissions_expr <= float(carbon_cap_kg), "NetworkCarbonBudget"

    model.solve(PULP_CBC_CMD(msg=solver_msg))
    status = LpStatus[model.status]

    if status != "Optimal":
        return {
            "status": status,
            "total_cost": np.nan,
            "total_emissions_kg": np.nan,
            "objective_with_carbon_price": np.nan,
            "flows": pd.DataFrame(index=loc, columns=loc, dtype=float),
            "plants": pd.DataFrame(index=loc, columns=size, dtype=float),
        }

    flow_df = pd.DataFrame(
        {
            j: [value(flow[(i, j)]) for i in loc]
            for j in loc
        },
        index=loc
    )

    plant_df = pd.DataFrame(
        {
            s: [value(plant[(i, s)]) for i in loc]
            for s in size
        },
        index=loc
    )

    fixed_cost_value = sum(
        float(total_fixed_costs.loc[i, s]) * 1000 * value(plant[(i, s)])
        for i in loc for s in size
    )

    operating_cost_value = sum(
        float(scenario_total_costs.loc[i, j]) * value(flow[(i, j)])
        for i in loc for j in loc
    )

    emissions_value = sum(
        float(carbon_emissions.loc[i, j]) * value(flow[(i, j)])
        for i in loc for j in loc
    )

    return {
        "status": status,
        "total_cost": fixed_cost_value + operating_cost_value,
        "total_emissions_kg": emissions_value,
        "objective_with_carbon_price": value(model.objective),
        "flows": flow_df,
        "plants": plant_df,
    }


## 4. Baseline economic optimization

The baseline minimizes economic cost **without imposing a hard lead-time cutoff or carbon price**.  
This provides the reference solution for all later experiments.


In [ ]:
baseline = solve_supply_chain()

if baseline["status"] != "Optimal":
    raise RuntimeError(f"Baseline model failed: {baseline['status']}")

print("Status:", baseline["status"])
print(f"Baseline monthly network cost: ${baseline['total_cost']:,.2f}")
print(f"Baseline CO2 emissions: {baseline['total_emissions_kg']/1000:,.2f} tCO2")

print("\nPlant-capacity decisions:")
display(baseline["plants"])

print("\nOptimal inter-country flows:")
display(baseline["flows"])

In [ ]:
origin_output = baseline["flows"].sum(axis=1) / 1_000_000

fig, ax = plt.subplots(figsize=(8, 4.5))
origin_output.plot(kind="bar", ax=ax)
ax.set_title("Baseline Production Allocation")
ax.set_xlabel("Manufacturing country")
ax.set_ylabel("Production (million units/month)")
plt.tight_layout()
plt.show()


## 5. Minimum-emission network

To define the lower endpoint of the Pareto frontier, the same network is solved with **CO₂ emissions as the sole objective**.

This is not proposed as the practical operating solution; it identifies the technically feasible decarbonization limit of the current network.


In [ ]:
def solve_minimum_emissions(solver_msg=False):
    model = LpProblem("Minimum_Emission_Network", LpMinimize)

    flow = LpVariable.dicts(
        "flow",
        [(i, j) for i in loc for j in loc],
        lowBound=0,
        cat="Continuous"
    )

    plant = LpVariable.dicts(
        "plant",
        [(i, s) for i in loc for s in size],
        cat="Binary"
    )

    emissions_expr = lpSum(
        float(carbon_emissions.loc[i, j]) * flow[(i, j)]
        for i in loc for j in loc
    )

    model += emissions_expr

    for j in loc:
        model += lpSum(flow[(i, j)] for i in loc) == float(demand.loc[j, "Demand"])

    for i in loc:
        model += (
            lpSum(flow[(i, j)] for j in loc)
            <= lpSum(
                float(capacity.loc[i, s]) * 1000 * plant[(i, s)]
                for s in size
            )
        )
        model += lpSum(plant[(i, s)] for s in size) <= 1

    model.solve(PULP_CBC_CMD(msg=solver_msg))
    status = LpStatus[model.status]

    if status != "Optimal":
        raise RuntimeError(f"Minimum-emission model failed: {status}")

    minimum_emissions = value(emissions_expr)

    if minimum_emissions is None or not np.isfinite(minimum_emissions):
        raise ValueError(f"Invalid minimum-emission value: {minimum_emissions}")

    return float(minimum_emissions)


minimum_emissions_kg = solve_minimum_emissions()
baseline_emissions_kg = float(baseline["total_emissions_kg"])

print(f"Minimum technically feasible emissions: {minimum_emissions_kg/1000:,.2f} tCO2")
print(f"Cost-optimal baseline emissions: {baseline_emissions_kg/1000:,.2f} tCO2")
print(
    f"Maximum technically feasible reduction: "
    f"{100*(1-minimum_emissions_kg/baseline_emissions_kg):.2f}%"
)


## 6. Cost–carbon Pareto frontier

The **ε-constraint method** is used to construct the frontier:

- keep economic cost as the objective,
- progressively tighten the allowable carbon budget,
- solve the MILP at each budget.

Every point is therefore the **least-cost solution available at that emission level**.


In [ ]:
carbon_caps = np.linspace(minimum_emissions_kg, baseline_emissions_kg, 15)

pareto_records = []

for cap_kg in carbon_caps:
    result = solve_supply_chain(carbon_cap_kg=float(cap_kg))

    if result["status"] == "Optimal":
        pareto_records.append({
            "Carbon Cap (tCO2)": cap_kg / 1000,
            "Cost ($/month)": result["total_cost"],
            "Emissions (tCO2)": result["total_emissions_kg"] / 1000,
            "Cost Premium (%)":
                100 * (result["total_cost"] / baseline["total_cost"] - 1),
            "CO2 Reduction (%)":
                100 * (1 - result["total_emissions_kg"] / baseline_emissions_kg),
        })

pareto_df = pd.DataFrame(pareto_records)

if pareto_df.empty:
    raise RuntimeError("No feasible Pareto solutions were generated.")

display(pareto_df.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(
    pareto_df["CO2 Reduction (%)"],
    pareto_df["Cost Premium (%)"],
    marker="o"
)
ax.set_title("Cost–Carbon Pareto Frontier")
ax.set_xlabel("CO2 reduction from cost-optimal baseline (%)")
ax.set_ylabel("Increase in monthly network cost (%)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Identify a normalized best-compromise point (closest to the utopia point:
# maximum emission reduction and minimum cost premium).

cost_norm = (
    pareto_df["Cost Premium (%)"] - pareto_df["Cost Premium (%)"].min()
) / max(
    pareto_df["Cost Premium (%)"].max() - pareto_df["Cost Premium (%)"].min(),
    1e-12
)

reduction_norm = (
    pareto_df["CO2 Reduction (%)"] - pareto_df["CO2 Reduction (%)"].min()
) / max(
    pareto_df["CO2 Reduction (%)"].max() - pareto_df["CO2 Reduction (%)"].min(),
    1e-12
)

distance_to_utopia = np.sqrt(cost_norm**2 + (1 - reduction_norm)**2)
compromise_idx = distance_to_utopia.idxmin()
compromise = pareto_df.loc[compromise_idx]

print("Normalized compromise solution:")
display(compromise.to_frame("Value").round(3))


## 7. Carbon-price policy scenarios

Instead of imposing an emissions cap, a policymaker may internalize emissions through a carbon price.

The model applies:

\[
\text{Carbon cost} =
p_c \times \frac{E_{\mathrm{kg}}}{1000}
\]

where \(p_c\) is in **$/tCO₂**.

The sweep below evaluates whether threshold effects occur—i.e., whether the optimal network changes abruptly once carbon becomes sufficiently expensive.


In [ ]:
carbon_prices = [0, 25, 50, 75, 100, 150, 200]

carbon_price_records = []
carbon_price_solutions = {}

for price in carbon_prices:
    result = solve_supply_chain(carbon_price_per_ton=price)
    carbon_price_solutions[price] = result

    if result["status"] == "Optimal":
        carbon_price_records.append({
            "Carbon Price ($/tCO2)": price,
            "Network Cost excluding carbon ($/month)": result["total_cost"],
            "Emissions (tCO2)": result["total_emissions_kg"] / 1000,
            "CO2 Reduction (%)":
                100 * (1 - result["total_emissions_kg"] / baseline_emissions_kg),
        })

carbon_price_df = pd.DataFrame(carbon_price_records)
display(carbon_price_df.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(
    carbon_price_df["Carbon Price ($/tCO2)"],
    carbon_price_df["CO2 Reduction (%)"],
    marker="o"
)
ax.set_title("Decarbonization Response to Carbon Pricing")
ax.set_xlabel("Carbon price ($/tCO2)")
ax.set_ylabel("CO2 reduction vs baseline (%)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Service-level feasibility analysis

The supplied delivery-deadline matrix uses a **30-day limit**. Applying that as a hard route cutoff makes the network infeasible because the remaining short-lead-time routes do not provide enough aggregate capacity to satisfy all demand.

Instead of hiding this infeasibility, the project treats it as a design question:

> **What is the minimum uniform lead-time allowance that makes the network feasible?**

This is a useful resilience/service-level insight.


In [ ]:
candidate_lead_limits = sorted(np.unique(delivery_times.to_numpy().astype(float)))

lead_records = []

for limit in candidate_lead_limits:
    result = solve_supply_chain(max_lead_days=float(limit))
    lead_records.append({
        "Maximum Allowed Lead Time (days)": limit,
        "Status": result["status"],
        "Cost ($/month)": result["total_cost"],
        "Emissions (tCO2)":
            result["total_emissions_kg"] / 1000
            if result["status"] == "Optimal" else np.nan,
    })

lead_df = pd.DataFrame(lead_records)
display(lead_df)

feasible_lead_df = lead_df[lead_df["Status"] == "Optimal"]

if not feasible_lead_df.empty:
    minimum_feasible_lead = feasible_lead_df.iloc[0]["Maximum Allowed Lead Time (days)"]
    print(f"Minimum feasible uniform lead-time allowance: {minimum_feasible_lead:.0f} days")
else:
    print("No feasible lead-time threshold found in the supplied route matrix.")


## 9. Capacity and demand-growth robustness

The network has finite installed capacity. A useful robustness question is:

> **How much uniform demand growth can the current candidate network absorb before no feasible plant configuration exists?**

The analysis first computes the theoretical capacity headroom and then verifies feasibility with the MILP.


In [ ]:
theoretical_growth_limit = maximum_capacity / total_demand - 1

print(f"Theoretical uniform demand-growth limit: {theoretical_growth_limit*100:.2f}%")

growth_tests = [0.00, 0.01, 0.02, theoretical_growth_limit, theoretical_growth_limit + 0.001]

growth_records = []

for growth in growth_tests:
    scenario_demand = demand.copy()
    scenario_demand["Demand"] = demand["Demand"] * (1 + growth)

    result = solve_supply_chain(demand_df=scenario_demand)

    growth_records.append({
        "Demand Growth (%)": growth * 100,
        "Status": result["status"],
        "Cost ($/month)": result["total_cost"],
        "Emissions (tCO2)":
            result["total_emissions_kg"] / 1000
            if result["status"] == "Optimal" else np.nan,
    })

growth_df = pd.DataFrame(growth_records)
display(growth_df.round(3))


## 10. Freight-cost disruption stress test

To represent a severe logistics shock, international freight costs are increased **3×** and the network is re-optimized.

The experiment measures:

- total cost exposure,
- emissions response,
- and shifts in production allocation.


In [ ]:
freight_shock = solve_supply_chain(freight_multiplier=3.0)

if freight_shock["status"] != "Optimal":
    raise RuntimeError(f"Freight-shock model failed: {freight_shock['status']}")

stress_summary = pd.DataFrame([
    {
        "Scenario": "Baseline",
        "Cost ($/month)": baseline["total_cost"],
        "Emissions (tCO2)": baseline["total_emissions_kg"] / 1000,
    },
    {
        "Scenario": "3× Freight Cost",
        "Cost ($/month)": freight_shock["total_cost"],
        "Emissions (tCO2)": freight_shock["total_emissions_kg"] / 1000,
    },
])

stress_summary["Cost Change (%)"] = (
    stress_summary["Cost ($/month)"] / baseline["total_cost"] - 1
) * 100

stress_summary["CO2 Change (%)"] = (
    stress_summary["Emissions (tCO2)"] /
    (baseline["total_emissions_kg"] / 1000) - 1
) * 100

display(stress_summary.round(3))

In [ ]:
production_comparison = pd.DataFrame({
    "Baseline": baseline["flows"].sum(axis=1) / 1_000_000,
    "3× Freight Cost": freight_shock["flows"].sum(axis=1) / 1_000_000,
})

ax = production_comparison.plot(kind="bar", figsize=(8, 4.8))
ax.set_title("Production Reallocation under Freight-Cost Disruption")
ax.set_xlabel("Manufacturing country")
ax.set_ylabel("Production (million units/month)")
plt.tight_layout()
plt.show()

display(production_comparison.round(3))


## 11. Consolidated research findings

This section automatically extracts the main quantitative findings so that claims used in a report or CV remain traceable to model outputs.


In [ ]:
max_reduction_row = pareto_df.loc[pareto_df["CO2 Reduction (%)"].idxmax()]

first_carbon_response = carbon_price_df[
    carbon_price_df["CO2 Reduction (%)"] > 1e-6
]

if not first_carbon_response.empty:
    carbon_threshold = float(first_carbon_response.iloc[0]["Carbon Price ($/tCO2)"])
else:
    carbon_threshold = np.nan

summary = pd.DataFrame([
    {
        "Metric": "Baseline monthly network cost",
        "Value": f"${baseline['total_cost']:,.2f}",
    },
    {
        "Metric": "Baseline network emissions",
        "Value": f"{baseline['total_emissions_kg']/1000:,.2f} tCO2",
    },
    {
        "Metric": "Maximum modeled CO2 reduction",
        "Value": f"{max_reduction_row['CO2 Reduction (%)']:.2f}%",
    },
    {
        "Metric": "Cost premium at maximum modeled CO2 reduction",
        "Value": f"{max_reduction_row['Cost Premium (%)']:.2f}%",
    },
    {
        "Metric": "Minimum feasible uniform route lead-time",
        "Value": f"{minimum_feasible_lead:.0f} days",
    },
    {
        "Metric": "Theoretical uniform demand-growth headroom",
        "Value": f"{theoretical_growth_limit*100:.2f}%",
    },
    {
        "Metric": "Cost increase under 3× freight shock",
        "Value": f"{(freight_shock['total_cost']/baseline['total_cost']-1)*100:.2f}%",
    },
    {
        "Metric": "First tested carbon price changing the network",
        "Value": (
            f"${carbon_threshold:.0f}/tCO2"
            if np.isfinite(carbon_threshold) else "No response in tested range"
        ),
    },
])

display(summary)


## 12. Research interpretation and limitations

### What this project demonstrates

- **MILP formulation** of a five-country capacitated supply-chain network.
- Simultaneous treatment of facility, production, freight, storage, demand, and capacity decisions.
- **Multi-objective cost–carbon optimization** through an ε-constraint Pareto frontier.
- **Carbon-price policy analysis** with explicit unit conversion to $/tCO₂.
- **Service-level feasibility analysis**, including identification of an infeasible 30-day hard deadline.
- **Capacity robustness analysis** under demand growth.
- **Disruption stress testing** under a 3× freight-cost shock.

### Limitations

The study remains a stylized deterministic network model. It does not yet include:

- stochastic demand distributions,
- inventory dynamics across multiple time periods,
- supplier failure probabilities,
- endogenous transport-mode selection,
- nonlinear economies of scale,
- or uncertainty in emission factors.

These limitations provide clear directions for future extensions such as **stochastic programming, robust optimization, or Monte Carlo scenario generation**.



## 13. CV / GitHub positioning

### Recommended project title

**Sustainable Multi-Objective Supply Chain Network Optimization under Cost, Carbon & Disruption Uncertainty**

### CV bullet template

After running all cells successfully, use only the values generated by the notebook:

- Formulated a **five-country MILP supply-chain network** in PuLP integrating plant capacity, production, freight, storage, demand, service-level and CO₂ constraints.
- Built a **cost–carbon Pareto framework with carbon-price, capacity and disruption stress tests**, quantifying decarbonization cost and network robustness.

For the final CV, replace generic phrases such as “quantifying decarbonization cost” with the **actual percentages produced in Section 11**.
